In [1]:
START_DATE = "2025-03-20"  #  "2025-03-19"
END_DATE = "2025-09-30"

TRIALS_COUNT = 150
N_JOBS = 14
ZONES_DEFAULT_TO_SPAIN = True

In [2]:
import sys
import os
import pandas as pd
import numpy as np
import wandb

In [3]:
FILE_DIR = os.getcwd()
# parent directory and src directory
SRC_DIR = os.path.join(os.path.dirname(FILE_DIR), "src")
DATA_DIR = os.path.join(os.path.dirname(FILE_DIR), "data")
print(f"Adding src directory to sys.path: {SRC_DIR}")
# Add the src directory to sys.path
sys.path.append(SRC_DIR)

Adding src directory to sys.path: /home/einsunza/Desktop/work/mibel-simulator/src


In [4]:
from mibel_simulator.clearing_process import clear_OMIE_market
import mibel_simulator.columns as cols

In [5]:
DET_FILEPATH_PATTERN = os.path.join(DATA_DIR, "DET/DET_{YYYYMMDD}.1")
CAB_FILEPATH_PATTERN = os.path.join(DATA_DIR, "CAB/CAB_{YYYYMMDD}.1")
CAPACIDAD_INTER_PBC_FILEPATH_PATTERN = os.path.join(
    DATA_DIR, "capacidad_inter_pbc/capacidad_inter_pbc_{YYYYMMDD}.1"
)
MARGINALPDBC_FILEPATH = os.path.join(DATA_DIR, "marginalpdbc.parquet")
PRICE_FRANCE_FILEPATH = os.path.join(DATA_DIR, "price_france.parquet")

CLEARED_DET_CAB_DATE_FILEPATH_PATTERN = os.path.join(
    DATA_DIR, "results/cleared_det_cab_date/cleared_det_cab_date_{YYYYMMDD}.parquet"
)

In [6]:
price_france_df = pd.read_parquet(PRICE_FRANCE_FILEPATH)
marginalpdbc = pd.read_parquet(MARGINALPDBC_FILEPATH)

In [7]:
failed_jobs = [
    "2025-03-30 00:00:00",
    "2025-04-21 00:00:00",
    "2025-04-30 00:00:00",
    "2025-05-06 00:00:00",
    "2025-05-30 00:00:00",
    "2025-05-31 00:00:00",
    "2025-06-01 00:00:00",
    "2025-07-26 00:00:00",
    "2025-08-13 00:00:00",
    "2025-08-16 00:00:00",
    "2025-08-17 00:00:00",
    "2025-08-24 00:00:00",
    "2025-08-30 00:00:00",
    "2025-08-31 00:00:00",
    "2025-09-14 00:00:00",
    "2025-09-18 00:00:00",
    "2025-09-20 00:00:00",
    "2025-09-23 00:00:00",
    "2025-09-24 00:00:00",
    "2025-09-25 00:00:00",
    "2025-09-27 00:00:00",
    "2025-09-28 00:00:00",
]
date_range = pd.to_datetime(failed_jobs)

In [8]:
# date_range = pd.date_range(start=START_DATE, end=END_DATE, freq="D")

for date in date_range:
    try:
        print(f"Processing date: {date.strftime('%Y-%m-%d')}")
        det_date = DET_FILEPATH_PATTERN.replace("{YYYYMMDD}", date.strftime("%Y%m%d"))
        cab_date = CAB_FILEPATH_PATTERN.replace("{YYYYMMDD}", date.strftime("%Y%m%d"))
        capacidad_inter_date = CAPACIDAD_INTER_PBC_FILEPATH_PATTERN.replace(
            "{YYYYMMDD}", date.strftime("%Y%m%d")
        )
        price_france_date = price_france_df[price_france_df[cols.DATE_SESION] == date]
        marginalpdbc_date = marginalpdbc[marginalpdbc.dat_sesion == date]
        cleared_det_cab_date_filepath = CLEARED_DET_CAB_DATE_FILEPATH_PATTERN.replace(
            "{YYYYMMDD}", date.strftime("%Y%m%d")
        )
        wandb.init(
            project="omie-clearing-process-260117",
            name=f"clearing-{date.strftime('%Y-%m-%d')}",
            config={
                "date": str(date),
                "trials_count": TRIALS_COUNT,
                "n_jobs": N_JOBS,
                "zones_default_to_spain": ZONES_DEFAULT_TO_SPAIN,
            },
            reinit=True,
        )

        results = clear_OMIE_market(
            det_date=det_date,
            cab_date=cab_date,
            capacidad_inter_date=capacidad_inter_date,
            price_france_date=price_france_date,
            trials_count=TRIALS_COUNT,
            zones_default_to_spain=ZONES_DEFAULT_TO_SPAIN,
            n_jobs=N_JOBS,
        )

        clearing_prices = results["clearing_prices"]
        model_trial_info = results["model_trial_info"]
        trials_df = results["trials_df"]
        calculated_spain_clearing_prices = (
            clearing_prices.query(f'{cols.CAT_PAIS} == "ES"')
            .set_index(f"{cols.INT_PERIODO}")
            .sort_index()["float_cleared_price"]
        )
        calculated_portugal_clearing_prices = (
            clearing_prices.query(f'{cols.CAT_PAIS} == "PT"')
            .set_index(f"{cols.INT_PERIODO}")
            .sort_index()["float_cleared_price"]
        )
        real_spain_clearing_prices = marginalpdbc_date.set_index("period").sort_index()[
            "precio_es"
        ]
        real_portugal_clearing_prices = marginalpdbc_date.set_index(
            "period"
        ).sort_index()["precio_pt"]

        spain_clearing_price_mae = (
            (calculated_spain_clearing_prices - real_spain_clearing_prices).abs().mean()
        )
        spain_clearing_price_mae_rounded = np.round(spain_clearing_price_mae, 2)
        portugal_clearing_price_mae = (
            (calculated_portugal_clearing_prices - real_portugal_clearing_prices)
            .abs()
            .mean()
        )
        portugal_clearing_price_mae_rounded = np.round(portugal_clearing_price_mae, 2)

        # transmissions

        calculated_spain_portugal_transmissions = results[
            "spain_portugal_transmissions"
        ].sort_index()

        # iterations to success

        matching_index_mask = np.isclose(
            trials_df["float_objective_value"],
            model_trial_info.float_objective_value,
            rtol=1e-9,
            atol=1e-12,
        )
        if not matching_index_mask.any():
            successful_trial_count = None
        else:
            matching_index = trials_df.index[matching_index_mask][0]
            successful_trial_count = matching_index + 1

        cleared_det_cab_date = results["cleared_det_cab_date"]
        cleared_det_cab_date.to_parquet(cleared_det_cab_date_filepath)

        wandb.log(
            {
                "date_str": str(date),
                "date": date,
                "trials_count": TRIALS_COUNT,
                "successful_trial_count": successful_trial_count,
                "objective_value": model_trial_info.float_objective_value,
                "mic_respected": model_trial_info.bool_is_mic_respected,
                "scos_with_mic_count": model_trial_info.int_mic_scos_count,
                "spain_clearing_price_mae": spain_clearing_price_mae,
                "spain_clearing_price_mae_rounded": spain_clearing_price_mae_rounded,
                "portugal_clearing_price_mae": portugal_clearing_price_mae,
                "portugal_clearing_price_mae_rounded": portugal_clearing_price_mae_rounded,
                "calculated_spain_portugal_transmissions": wandb.Table(
                    data=calculated_spain_portugal_transmissions.reset_index()
                ),
                "calculated_spain_clearing_prices": wandb.Table(
                    data=calculated_spain_clearing_prices.reset_index()
                ),
                "calculated_portugal_clearing_prices": wandb.Table(
                    data=calculated_portugal_clearing_prices.reset_index()
                ),
                "real_spain_clearing_prices": wandb.Table(
                    data=real_spain_clearing_prices.reset_index()
                ),
                "real_portugal_clearing_prices": wandb.Table(
                    data=real_portugal_clearing_prices.reset_index()
                ),
            }
        )
        wandb.finish()

    except Exception as e:
        print(f"Error processing date {date.strftime('%Y-%m-%d')}: {e}")
        continue

Processing date: 2025-03-30


wandb: Currently logged in as: einsunza (iit-energy-prediction) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Period 25 size is less than 0.1 times the size of the other periods, dropping it. This is a typical issue with OMIE det files. File: /home/einsunza/Desktop/work/mibel-simulator/data/DET/DET_20250330.1


24 2390.125
ERROR: Rule failed when generating expression for Constraint
c_Congestion_Spain_Portugal with index 24: ValueError: Error retrieving
immutable Param value (p_congestion_spain_portugal_exportacion[24]):
            The Param value is undefined and no default value is specified.
ERROR: Constructing component 'c_Congestion_Spain_Portugal' from data=None
failed:
        ValueError: Error retrieving immutable Param value
        (p_congestion_spain_portugal_exportacion[24]):
    	The Param value is undefined and no default value is specified.
Error processing date 2025-03-30: Error retrieving immutable Param value (p_congestion_spain_portugal_exportacion[24]):
	The Param value is undefined and no default value is specified.
Processing date: 2025-04-21


Period 25 size is less than 0.1 times the size of the other periods, dropping it. This is a typical issue with OMIE det files. File: /home/einsunza/Desktop/work/mibel-simulator/data/DET/DET_20250421.1


...

Processing date: 2025-09-28


Period 25 size is less than 0.1 times the size of the other periods, dropping it. This is a typical issue with OMIE det files. File: /home/einsunza/Desktop/work/mibel-simulator/data/DET/DET_20250928.1


28 2668.5416666666665


The following id_unidad in det_cab_date were not found in unidades_zona and have been assigned to ES: ['BESC01' 'NXVD509' 'NXVD508' 'NXVD510' 'CEVD489' 'AAXRGDL' 'EGVD529'
 'VGERE62' 'NEURM20' 'NEURM21' 'IGVD256' 'UPACAC' 'UPACAV' 'GSVD193'
 'GSVD194' 'GSVD196' 'GSVD197' 'GSVD198' 'GSVD189' 'GNVD292']
/home/einsunza/Desktop/work/mibel-simulator/src/mibel_simulator/clearing_process.py:730: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  trials_df = pd.concat([trials_df] + results, ignore_index=True)


objective_value,▁
portugal_clearing_price_mae,▁
portugal_clearing_price_mae_rounded,▁
scos_with_mic_count,▁
spain_clearing_price_mae,▁
spain_clearing_price_mae_rounded,▁
successful_trial_count,▁
trials_count,▁
date,2025-09-28T00:00:00
date_str,2025-09-28 00:00:00
mic_respected,True


# TEMP

In [ ]:
(40 - 35) * 100 + (41 - 35) * 110 - 1000 + (30 - 35) * 50

-90

In [ ]:
40 * 100 + 41 * 110  # + 30*50

8510

In [ ]:
35 * 100 + 35 * 110 + 1000

8350